In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [4]:
raw_shark_df = pd.read_csv("../data/clean_data/clean_data.csv")
shark_df = raw_shark_df.drop(["type"], axis=1)

print(shark_df.head(), "\n")

nonfatal_df = shark_df[shark_df["fatal"] == "N"]
fatal_df = shark_df[shark_df["fatal"] == "Y"]

# compute overall statistics
total_attacks = len(shark_df)
total_fatal = len(fatal_df)
total_nonfatal = len(nonfatal_df)
prop_fatal = total_fatal / total_attacks

print(f"Total number of attacks: {total_attacks}")
print(f"Number of fatal attacks: {total_fatal}")
print(f"Number of nonfatal attacks: {total_nonfatal}")
print(f"Overall fatality rate: {prop_fatal:.2%}")

             country            state  activity  sex   age fatal       time  \
0            bahamas      staniel cay  swimming    m  12.0     N  afternoon   
1                usa          florida  swimming    f  17.0     N    morning   
2          australia  new south wales  swimming    f  35.0     N    morning   
3  galapogos islands  santa fe island  swimming    f   NaN     N  afternoon   
4                usa          florida  swimming  NaN   NaN     N    morning   

       species  year  month  
0          NaN   NaN    6.0  
1          NaN   NaN    6.0  
2  great white   NaN    6.0  
3          NaN   NaN    6.0  
4         bull   NaN    6.0   

Total number of attacks: 5164
Number of fatal attacks: 1273
Number of nonfatal attacks: 3891
Overall fatality rate: 24.65%


In [5]:
country_counts = shark_df.groupby("country").size().to_frame(name="count").reset_index()
country_counts.sort_values(by="count", ascending=False, inplace=True)
country_counts["prop_total"] = country_counts["count"] / total_attacks
top_10_countries = country_counts.head(10)

print(f"Top 10 Countries account for {top_10_countries["prop_total"].sum()}")
top_10_countries

Top 10 Countries account for 0.7937645236250968


,country,count,prop_total
158,usa,1994,0.386135
8,australia,1084,0.209915
131,south africa,413,0.079977
113,papua new guinea,110,0.021301
9,bahamas,109,0.021108
15,brazil,100,0.019365
105,new zealand,95,0.018397
94,mexico,74,0.014330
103,new caledonia,60,0.011619
48,fiji,60,0.011619


In [6]:
filtered_df = shark_df[shark_df["country"].isin(top_10_countries["country"])]

nonfatal_df = filtered_df[filtered_df["fatal"] == "N"]
fatal_df = filtered_df[filtered_df["fatal"] == "Y"]

# compute statistics
total_attacks = len(filtered_df)
total_fatal = len(fatal_df)
total_nonfatal = len(nonfatal_df)
prop_fatal = total_fatal / total_attacks

print("FILTERED TO TOP 10 COUNTRIES:")
print(f"Total number of attacks: {total_attacks}")
print(f"Number of fatal attacks: {total_fatal}")
print(f"Number of nonfatal attacks: {total_nonfatal}")
print(f"Overall fatality rate: {prop_fatal:.2%}")

FILTERED TO TOP 10 COUNTRIES:
Total number of attacks: 4099
Number of fatal attacks: 780
Number of nonfatal attacks: 3319
Overall fatality rate: 19.03%


In [7]:
nonfatal_df = filtered_df[filtered_df["fatal"] == "N"]
fatal_df = filtered_df[filtered_df["fatal"] == "Y"]

state_counts1 = fatal_df.groupby("state").size().to_frame(name="fatal_count").reset_index()
state_counts2 = nonfatal_df.groupby("state").size().to_frame(name="nonfatal_count").reset_index()

state_counts = state_counts2.merge(state_counts1, on="state", how="left").fillna(0)
state_counts["total_count"] = state_counts["fatal_count"] + state_counts["nonfatal_count"]
state_counts["fatality_rate"] = state_counts["fatal_count"] / state_counts["total_count"]

state_counts.sort_values("total_count", ascending=False, inplace=True)

print(len(state_counts))
state_counts.head()

164


,state,nonfatal_count,fatal_count,total_count,fatality_rate
43,florida,946,41.0,987.0,0.041540
95,new south wales,279,102.0,381.0,0.267717
54,hawaii,229,47.0,276.0,0.170290
115,queensland,186,72.0,258.0,0.279070
22,california,195,20.0,215.0,0.093023


* Filter down to only states with 30 or more attacks

In [10]:
keep_states = list(state_counts[state_counts["total_count"] >= 30]["state"])
top_states_df = filtered_df[filtered_df["state"].isin(keep_states)]

print(f"Num attacks in states with >=30 attacks: {len(top_states_df)}")
print(f"Total attacks in parent dataset: {len(shark_df)}")
print(f"Percent of total data kept: {len(top_states_df) / len(shark_df)}")
print(f"Number of unique states remaining: {len(top_states_df["state"].unique())}")

top_states_df.head()

# with open("top_states.txt", "w") as f:
#     for state in top_states_df["state"].unique():
#         f.write(state+"\n")

Num attacks in states with >=30 attacks: 3420
Total attacks in parent dataset: 5164
Percent of total data kept: 0.6622773044151821
Number of unique states remaining: 20


,country,state,activity,sex,age,fatal,time,species,year,month
1,usa,florida,swimming,f,17.0,N,morning,NaN,NaN,6.0
2,australia,new south wales,swimming,f,35.0,N,morning,great white,NaN,6.0
4,usa,florida,swimming,NaN,NaN,N,morning,bull,NaN,6.0
5,australia,western australia,dive fishing,m,35.0,Y,morning,great white,NaN,6.0
8,australia,new south wales,surfing/bodyboarding,NaN,20.0,N,evening,bull,NaN,5.0


In [9]:
top_states_df["state"].unique()

<StringArray>
[              'florida',       'new south wales',     'western australia',
                'hawaii',            'queensland',       'south australia',
            'california',                 'texas',         'kwazulu-natal',
        'south carolina',        'north carolina',              'victoria',
          'south island',          'north island',            'new jersey',
            'pernambuco', 'western cape province', 'eastern cape province',
                'oregon',         'torres strait']
Length: 20, dtype: str

In [17]:
def map_hemisphere(state):

    hemisphere_map = {
        "northern": ["florida", "hawaii", "california", "texas", "south carolina", "north carolina", "new jersey", "oregon"],
        "southern": ["new south wales", "western australia", "queensland", "south australia" , "kwazulu-natal", "victoria"
            "south island", "north island", "pernambuco", "western cape province", "eastern cape province", "torres strait"]
    }

    for hemi, state_list in hemisphere_map.items():
        if state in state_list:
            return hemi
    return np.nan

top_states_df["hemisphere"] = top_states_df["state"].apply(lambda x: map_hemisphere(x))

top_states_df

,country,state,activity,sex,age,fatal,time,species,year,month,hemisphere
1,usa,florida,swimming,f,17.0,N,morning,NaN,NaN,6.0,northern
2,australia,new south wales,swimming,f,35.0,N,morning,great white,NaN,6.0,southern
4,usa,florida,swimming,NaN,NaN,N,morning,bull,NaN,6.0,northern
5,australia,western australia,dive fishing,m,35.0,Y,morning,great white,NaN,6.0,southern
8,australia,new south wales,surfing/bodyboarding,NaN,20.0,N,evening,bull,NaN,5.0,southern
...,...,...,...,...,...,...,...,...,...,...,...
5154,usa,hawaii,NaN,f,NaN,N,NaN,NaN,NaN,NaN,northern
5158,australia,new south wales,swimming,m,NaN,Y,NaN,nurse,NaN,NaN,southern
5159,australia,western australia,diving,m,NaN,Y,NaN,NaN,NaN,NaN,southern
5160,australia,western australia,diving,m,NaN,Y,NaN,NaN,NaN,NaN,southern


In [22]:
def map_season(row):

    north_map = {
        "spring": [3, 4, 5],
        "summer": [6, 7, 8],
        "fall": [9, 10, 11],
        "winter": [12, 1, 2]
    }

    south_map = {
        "spring": [9, 10, 11],
        "summer": [12, 1, 2],
        "fall": [3, 4, 5],
        "winter": [6,7, 8]
    }

    try:
        if row.hemisphere == "northern":
            for season, month_list in north_map.items():
                if int(row.month) in month_list:
                    return season
        elif row.hemisphere == "southern":
            for season, month_list in south_map.items():
                if int(row.month) in month_list:
                    return season
    except:
        return np.nan


season_col = []
for row in top_states_df.itertuples():
    season_col.append(map_season(row))

top_states_df["season"] = season_col
top_states_df.head()

,country,state,activity,sex,age,fatal,time,species,year,month,hemisphere,season
1,usa,florida,swimming,f,17.0,N,morning,NaN,NaN,6.0,northern,summer
2,australia,new south wales,swimming,f,35.0,N,morning,great white,NaN,6.0,southern,winter
4,usa,florida,swimming,NaN,NaN,N,morning,bull,NaN,6.0,northern,summer
5,australia,western australia,dive fishing,m,35.0,Y,morning,great white,NaN,6.0,southern,winter
8,australia,new south wales,surfing/bodyboarding,NaN,20.0,N,evening,bull,NaN,5.0,southern,fall


In [23]:
top_states_df.to_csv("../data/clean_data/top_states.csv", index=False)